# 优化器类

上一章我们实现了自动微分：只需一行 `loss.backward()`，所有参数的梯度就自动计算完毕。但参数更新这一步，还是要我们手动写。随着网络层数增加，参数可能达到数百万，手动更新每一个显然不现实。

本章引入**优化器**（Optimizer）类，把参数更新这一步也自动化，完成整个框架最后一块拼图。

至此，框架三条链路的封装全部完成：

| 链路 | 负责的类 |
|------|----------|
| 前向传播 | **层**（Layer）|
| 梯度计算 | **张量**（Tensor）+ **损失函数**（Loss）|
| 参数更新 | **优化器**（Optimizer）|

In [1]:
from abc import ABC, abstractmethod

import numpy as np

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础层

本章为基础层新增一个属性：**参数列表 `parameters`**。

它返回本层所有需要参与梯度更新的张量。优化器通过这个接口，自动收集网络中所有层的参数，
而不需要知道每个层内部的具体结构。

In [3]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [4]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

**基础优化器**（Optimizer）也是一个抽象类，定义了参数更新的统一接口。

创建一个优化器需要两样东西：

* **参数列表 `parameters`**：所有需要更新的参数张量列表；
* **学习率 `lr`**：每次更新时沿梯度方向移动的步长比例。

核心方法 `step()` 执行一次参数更新，具体的更新规则由子类实现。

In [5]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    @abstractmethod
    def step(self):
        pass

## 数据

### 特征、标签

In [6]:
feature = Tensor([28.1, 58.0])
label = Tensor([165])

## 模型

### 线性层

线性层覆盖了参数列表 `parameters`，返回 `[self.weight, self.bias]`。
这样优化器就能自动找到需要更新的参数张量，不必知道层的内部结构。

In [7]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad * x.data
            self.bias.grad += np.sum(p.grad)

        p.gradient_fn = gradient_fn
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 均方误差损失函数

In [8]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

我们的第一个优化器类是**随机梯度下降优化器**（SGDOptimizer）。参数更新的规则非常简单：将每个参数减去乘以学习率的对应梯度。

$$
\theta \leftarrow \theta - \eta \cdot \frac{dL}{d\theta}
$$

其中 $\theta$ 代表任意参数（权重或偏置），$\eta$ 是学习率。

In [9]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

## 训练

### 超参数：学习率

In [10]:
LEARNING_RATE = 0.00001

### 建模

In [11]:
layer = Linear(2, 1)
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)
print(layer)

Linear[weight(1, 2); bias(1,)]


### 前向传播

In [12]:
prediction = layer(feature)
print(f'prediction: {prediction}')

prediction: Tensor([43.05])


### 计算损失

In [13]:
loss = loss_fn(prediction, label)
print(f'loss: {loss}')

loss: Tensor(14871.802500000002)


### 梯度计算

In [14]:
loss.backward()
print(f'weight.grad: {layer.weight.grad}')
print(f'bias.grad:   {layer.bias.grad}')

weight.grad: [[ -6853.59 -14146.2 ]]
bias.grad:   [-243.9]


### 反向传播

In [15]:
optimizer.step()
print(f'weight: {layer.weight}')
print(f'bias:   {layer.bias}')

weight: Tensor([[0.5685359 0.641462 ]])
bias:   Tensor([0.002439])


## 验证

### 推理

In [16]:
prediction = layer(feature)
print(f'prediction: {prediction}')

prediction: Tensor([53.18309379])


### 评估

In [17]:
loss = loss_fn(prediction, label)
print(f'loss: {loss}')

loss: Tensor(12503.020514375934)


损失从 `14871.8` 降到了 `12503.0`，下降了约 `16%`。

至此，我们的神经网络训练框架的三条数据链路已经完整。

## 课后练习

父节点集合 `parents` 和参数列表 `parameters` 的设计，使得计算图的遍历和反向传播都可以自动化完成。尝试把不同的张量从父节点集合和参数列表中移除，观察反向传播的结果如何？